In [ ]:
from ts2.data.cell_data_module import CellDataModule 
import yaml
from omegaconf import OmegaConf
import torch
import matplotlib.pyplot as plt
import einops
import math
import torchvision
from PIL import Image
import numpy as np
from typing import List, Tuple
import time

In [ ]:
cf = """
infra:
  seed: 10
data:
  set: scsrh
  parser:
    which: CachedCSVParser
    params:
      cached_parser_file:
        train: ../data/tsmeta/srh7v1_sc/55e0deb2_cell_train
        trainval: ../data/tsmeta/srh7v1_sc/55e0deb2_cell_trainval
        test_databank: ../data/tsmeta/srh7v1_sc/c191d6d5_cell40_train
        test: ../data/tsmeta/srh7v1_sc/c191d6d5_cell40_trainval
  transform:
    train:
      which: HistologyTransform
      params:
        base_aug_params:
          laser_noise_config: null
          get_third_channel_params:
            mode: get_third_channel_cellmask_passthrough
            subtracted_base: 0.07629394531
          mask_params:
            how_to_process: rgba #small_patch
            bg_fill: [9957.5889, 10057.9795]
          to_uint8: True
        strong_aug_params:
          aug_list: []
          aug_prob: 1
    trainval: ${.train}
  dataset:
    which: CellDataset # SingleLevelHierarchicalDataset
    which_process_read: cell_memmap
    params:
      common:
        data_root: /nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root2/
        num_transforms: 1
        balance_instance_class: False
      train: {}
      trainval: {}
  loader:
    params:
      common:
        pin_memory: False
        persistent_workers: True
        prefetch_factor: 12
        num_workers: 7
      train:
        batch_size: 256
        drop_last: True
        shuffle: True
      trainval:
        batch_size: 64
        drop_last: False
        shuffle: False
"""

cf = OmegaConf.create(yaml.safe_load(cf))

In [ ]:
dm = CellDataModule(config=cf)
dm.setup(stage="fit")

In [ ]:
dset = dm.trainval_dataset_

In [ ]:
len(dset)

In [ ]:
k = 529
pit_idx = torch.where(torch.tensor([i["label"]=="pituita" for i in dset.instances_]))[0]
pit_sample_idx = sorted(pit_idx[torch.randperm(len(pit_idx))][:k])
pit_data = [dset.__getitem__(i) for i in pit_sample_idx]

In [ ]:
all_images = torch.stack([einops.rearrange(s["image"], "1 c h w -> h w c") for s in pit_data])
all_images_transparent = torch.clone(all_images)
all_images[...,-1] = 255
all_images_transparent[...,-1] = ((all_images_transparent[...,-1].to(torch.int32)+255)/2).to(torch.uint8)

In [ ]:
all_im_tile = einops.rearrange(
    torch.nn.functional.pad(
    all_images, pad=(0,0,5,5,5,5), value=255),
    "(bh bw) h w c -> (bh h) (bw w) c", bh=int(math.sqrt(k)), bw=int(math.sqrt(k)))
all_images_transparent = einops.rearrange(
    torch.nn.functional.pad(
    all_images_transparent, pad=(0,0,5,5,5,5), value=255),
    "(bh bw) h w c -> (bh h) (bw w) c", bh=int(math.sqrt(k)), bw=int(math.sqrt(k)))

In [ ]:
fig, ax=plt.subplots(2,1,figsize=(20,40))
ax[0].imshow(all_im_tile)
ax[1].imshow(all_images_transparent)

ax[0].axis("off")
ax[1].axis("off")

# First iteration of masking collate

In [ ]:
rand_idxs_fg = torch.randperm(len(dm.train_dataset_))[:72]
fg_im = [dm.train_dataset_.__getitem__(idx) for idx in rand_idxs_fg]

In [ ]:
class SingleCellBlendedCollator():

    def __init__(self, kernel_size, sigma, min_val_sigma, strong_transforms):
        self.ks = kernel_size
        self.sigma = sigma
        self.transform = strong_transforms
        self.min_val_sigma = min_val_sigma

    @staticmethod
    def _get_gaussian_kernel1d(kernel_size: int, sigma: float, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        ksize_half = (kernel_size - 1) * 0.5

        x = torch.linspace(-ksize_half, ksize_half, steps=kernel_size, dtype=dtype, device=device)
        pdf = torch.exp(-0.5 * (x / sigma).pow(2))
        kernel1d = pdf / pdf.sum()

        return kernel1d
    @staticmethod
    def _get_gaussian_kernel2d(
        kernel_size: List[int], sigma: List[float], dtype: torch.dtype, device: torch.device
    ) -> torch.Tensor:
        kernel1d_x = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[0], sigma[0], dtype, device)
        kernel1d_y = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[1], sigma[1], dtype, device)
        kernel2d = torch.mm(kernel1d_y[:, None], kernel1d_x[None, :])
        return kernel2d

    @staticmethod
    def gaussian_blur(img: torch.Tensor, kernel_size: List[int], sigma: List[float]) -> torch.Tensor:
        kernel = SingleCellBlendedCollator._get_gaussian_kernel2d(kernel_size, sigma, dtype=img.dtype, device=img.device)
        kernel = kernel.expand(img.shape[-3], 1, kernel.shape[0], kernel.shape[1])
        # padding = (left, right, top, bottom)
        padding = [kernel_size[0] // 2, kernel_size[0] // 2, kernel_size[1] // 2, kernel_size[1] // 2]
        img = torch.nn.functional.pad(img, padding)
        img = torch.nn.functional.conv2d(img, kernel, groups=img.shape[-3])

        return img
        
        
        
        
    def blend_batch(self, fg_im, bg_im):
        blend_scale = (fg_im[:, -1, ...] / 255).to(float)
        #blend_scale = torchvision.transforms.functional.gaussian_blur(
        #    blend_scale, kernel_size=self.ks, sigma= self.sigma)
        blend_scale = torch.cat([
            self.gaussian_blur(bs.unsqueeze(0),
                          kernel_size=[self.ks, self.ks],
                          sigma=[s,s])
            for bs, s in zip(
                blend_scale,
                torch.randn((len(blend_scale))).abs() * self.sigma + 1.0e-6)
        ])

        min_in = einops.reduce(
            blend_scale.masked_fill(~fg_im[:, -1, ...].to(bool), float("inf")),
            "b h w -> b", "min")
        min_in = min_in * torch.maximum(
            1 - torch.rand((len(blend_scale))).abs() * self.min_val_sigma,
            torch.tensor(1.0e-6))
        nuclei_plus_mask = blend_scale > einops.rearrange(min_in, "b -> b 1 1")
        
        to_fill_idx = torch.where(nuclei_plus_mask)
        to_fill_value = min_in[to_fill_idx[0]]
        blend_scale[to_fill_idx[0], to_fill_idx[1],
                    to_fill_idx[2]] = to_fill_value

        blend_scale = blend_scale - einops.reduce(blend_scale,
                                                  "b h w -> b 1 1", "min")
        blend_scale = blend_scale / einops.reduce(blend_scale,
                                                  "b h w -> b 1 1", "max")
        blend_scale = torch.nan_to_num(blend_scale, nan=1.0)

        bg_im_cutout = torch.clone(bg_im[:, :-1, ...])

        bg_nu_mask = bg_im[:, -1, ...].to(bool).to(torch.uint8)
        bg_nu_mask3 = einops.repeat(bg_nu_mask,
                                    "b h w -> b c h w",
                                    c=bg_im_cutout.shape[1])
        bg_nu_fillval = (
            (bg_im_cutout * (1 - bg_nu_mask3)).sum(dim=(2, 3)) /
            einops.reduce(1 - bg_nu_mask, "b h w -> b 1", "sum")
        ).to(fg_im.dtype) # yapf: disable

        bg_nu_idx = torch.where(bg_nu_mask)
        bg_nu_fillval = bg_nu_fillval[bg_nu_idx[0]]
        bg_im_cutout[bg_nu_idx[0], :, bg_nu_idx[1],
                     bg_nu_idx[2]] = bg_nu_fillval

        reconstructed = (fg_im[:, :-1, ...] * blend_scale.unsqueeze(1) +
                         bg_im_cutout * (1 - blend_scale).unsqueeze(1)).to(
                             fg_im.dtype)
        nuclei_plus_mask = nuclei_plus_mask.to(torch.uint8) + (fg_im[:, -1, ...]/255).to(torch.uint8)
        return reconstructed, nuclei_plus_mask, blend_scale

    def __call__(self, raw_batch):
        all_images = torch.stack([i["image"]
                                  for i in raw_batch])  # b aug c h w
        assert all_images.shape[1] == 1
        fg = all_images[:, 0, ...]
        bg = torch.clone(all_images[:, 0, ...])
        bg = bg[torch.randperm(len(bg))]
        
        #fg_clone = torch.clone(fg[:, :3, ...])
        t0 = time.time()
        alt_fg, nuclei_plus_mask, blend_scale = self.blend_batch(fg, bg)
        t1 = time.time()
        print(t1-t0)
        fg = torch.stack([self.transform(i) for i in fg[:, :3, ...]])
        alt_fg = torch.stack([self.transform(i) for i in alt_fg])
        all_im = torch.stack((fg, alt_fg), dim=1)

        #torch.save({"blend_scale": blend_scale, "fg": fg, "bg": bg, "alt_fg": alt_fg, "fg_clean": fg_clone}, "out.pt")
        #exit(1)
        return {
            "image": all_im,
            "label": torch.stack([i["label"] for i in raw_batch]),
            "path": [i["path"] for i in raw_batch],
            "nuclei_plus_mask": nuclei_plus_mask,
            "blend_scale": blend_scale,
            "bg":bg,
        }


In [ ]:
class SingleCellBlendedCollator():

    def __init__(self, kernel_size, sigma, min_val_sigma, strong_transforms):
        self.ks = kernel_size
        self.sigma = sigma
        self.transform = strong_transforms
        self.min_val_sigma = min_val_sigma

    @staticmethod
    def _get_gaussian_kernel1d(kernel_size: int, sigma: float, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        ksize_half = (kernel_size - 1) * 0.5

        x = torch.linspace(-ksize_half, ksize_half, steps=kernel_size, dtype=dtype, device=device)
        pdf = torch.exp(-0.5 * (x / sigma).pow(2))
        kernel1d = pdf / pdf.sum()

        return kernel1d
    @staticmethod
    def _get_gaussian_kernel2d(
        kernel_size: List[int], sigma: List[float], dtype: torch.dtype, device: torch.device
    ) -> torch.Tensor:
        kernel1d_x = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[0], sigma[0], dtype, device)
        kernel1d_y = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[1], sigma[1], dtype, device)
        kernel2d = torch.mm(kernel1d_y[:, None], kernel1d_x[None, :])
        return kernel2d

    @staticmethod
    def gaussian_blur(img: torch.Tensor, kernel_size: List[int], sigma: List[float]) -> torch.Tensor:
        kernel = SingleCellBlendedCollator._get_gaussian_kernel2d(kernel_size, sigma, dtype=img.dtype, device=img.device)
        kernel = kernel.expand(img.shape[-3], 1, kernel.shape[0], kernel.shape[1])
        # padding = (left, right, top, bottom)
        padding = [kernel_size[0] // 2, kernel_size[0] // 2, kernel_size[1] // 2, kernel_size[1] // 2]
        img = torch.nn.functional.pad(img, padding)
        img = torch.nn.functional.conv2d(img, kernel, groups=img.shape[-3])

        return img
        
        
        
        
    def blend_batch(self, fg_im, bg_im):
        blend_scale = (fg_im[:, -1, ...] / 255).to(float)
        blend_scale = torch.cat([
            self.gaussian_blur(bs.unsqueeze(0),
                          kernel_size=[self.ks, self.ks],
                          sigma=[s,s])
            for bs, s in zip(
                blend_scale,
                torch.randn((len(blend_scale))).abs() * self.sigma + 1.0e-6)
        ])
        nuclei_plus_mask = blend_scale
        blend_scale = (blend_scale > 0.05).to(float)
        
        bg_im_cutout = bg_im[:, :-1, ...]
        bg_nu_mask = bg_im[:, -1, ...].to(bool).to(torch.uint8)
        bg_nu_mask3 = einops.repeat(bg_nu_mask,
                                    "b h w -> b c h w",
                                    c=bg_im_cutout.shape[1])
        bg_nu_fillval = (
            (bg_im_cutout * (1 - bg_nu_mask3)).sum(dim=(2, 3)) /
            einops.reduce(1 - bg_nu_mask, "b h w -> b 1", "sum")
        ).to(fg_im.dtype) # yapf: disable

        bg_nu_idx = torch.where(bg_nu_mask)
        bg_nu_fillval = bg_nu_fillval[bg_nu_idx[0]]
        bg_im_cutout[bg_nu_idx[0], :, bg_nu_idx[1],
                     bg_nu_idx[2]] = bg_nu_fillval

        reconstructed = (fg_im[:, :-1, ...] * blend_scale.unsqueeze(1) +
                         bg_im_cutout * (1 - blend_scale).unsqueeze(1)).to(
                             fg_im.dtype)
        nuclei_plus_mask = blend_scale.to(torch.uint8) + (fg_im[:, -1, ...]/255).to(torch.uint8)
        return reconstructed, nuclei_plus_mask, blend_scale

    def __call__(self, raw_batch):
        all_images = torch.stack([i["image"]
                                  for i in raw_batch])  # b aug c h w
        assert all_images.shape[1] == 1
        fg = all_images[:, 0, ...]
        bg = torch.clone(all_images[:, 0, ...])
        bg = bg[torch.randperm(len(bg))]
        
        #fg_clone = torch.clone(fg[:, :3, ...])
        t0 = time.time()
        alt_fg, nuclei_plus_mask, blend_scale = self.blend_batch(fg, bg)
        t1 = time.time()
        print(t1-t0)
        fg = torch.stack([self.transform(i) for i in fg[:, :3, ...]])
        alt_fg = torch.stack([self.transform(i) for i in alt_fg])
        all_im = torch.stack((fg, alt_fg), dim=1)

        #torch.save({"blend_scale": blend_scale, "fg": fg, "bg": bg, "alt_fg": alt_fg, "fg_clean": fg_clone}, "out.pt")
        #exit(1)
        return {
            "image": all_im,
            "label": torch.stack([i["label"] for i in raw_batch]),
            "path": [i["path"] for i in raw_batch],
            "nuclei_plus_mask": nuclei_plus_mask,
            "blend_scale": blend_scale,
            "bg":bg,
        }


In [ ]:
class SingleCellBlendedCollator():

    def __init__(self, kernel_size, sigma, min_val_sigma, strong_transforms):
        self.ks = kernel_size
        self.sigma = sigma
        self.transform = strong_transforms
        self.min_val_sigma = min_val_sigma

    @staticmethod
    def _get_gaussian_kernel1d(kernel_size: int, sigma: float, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        ksize_half = (kernel_size - 1) * 0.5

        x = torch.linspace(-ksize_half, ksize_half, steps=kernel_size, dtype=dtype, device=device)
        pdf = torch.exp(-0.5 * (x / sigma).pow(2))
        kernel1d = pdf / pdf.sum()

        return kernel1d
    @staticmethod
    def _get_gaussian_kernel2d(
        kernel_size: List[int], sigma: List[float], dtype: torch.dtype, device: torch.device
    ) -> torch.Tensor:
        kernel1d_x = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[0], sigma[0], dtype, device)
        kernel1d_y = SingleCellBlendedCollator._get_gaussian_kernel1d(kernel_size[1], sigma[1], dtype, device)
        kernel2d = torch.mm(kernel1d_y[:, None], kernel1d_x[None, :])
        return kernel2d

    @staticmethod
    def gaussian_blur(img: torch.Tensor, kernel_size: List[int], sigma: List[float]) -> torch.Tensor:
        kernel = SingleCellBlendedCollator._get_gaussian_kernel2d(kernel_size, sigma, dtype=img.dtype, device=img.device)
        kernel = kernel.expand(img.shape[-3], 1, kernel.shape[0], kernel.shape[1])
        # padding = (left, right, top, bottom)
        padding = [kernel_size[0] // 2, kernel_size[0] // 2, kernel_size[1] // 2, kernel_size[1] // 2]
        img = torch.nn.functional.pad(img, padding)
        img = torch.nn.functional.conv2d(img, kernel, groups=img.shape[-3])

        return img
        
        
        
        
    def blend_batch(self, fg_im, bg_im):
        where_bm = [torch.stack(torch.where(b)) for b in fg_im[:, -1, ...]>0]
        
        where_bm_min = [b.min(dim=1).values for b in where_bm]
        where_bm_max = [b.max(dim=1).values for b in where_bm]
        minrc = [(torch.randint(low=0,high=i[0],size=(1,)), torch.randint(low=0,high=i[1],size=(1,)) )for i in where_bm_min]
        maxrc = [(torch.randint(low=i[0]+1,high=64,size=(1,)), torch.randint(low=i[1]+1,high=64,size=(1,)))  for i in where_bm_max]

        blend_scale = torch.zeros_like(fg_im)
        for i, (mi, ma, fg, bg) in enumerate(zip(minrc, maxrc, fg_im, bg_im)):
            bg[:,mi[0]:ma[0], mi[1]: ma[1]] = fg[:,mi[0]:ma[0], mi[1]: ma[1]]
            blend_scale[i,:, mi[0]:ma[0], mi[1]: ma[1]] = 1
            
        return bg_im[:,:3,...], blend_scale

    def __call__(self, raw_batch):
        all_images = torch.stack([i["image"]
                                  for i in raw_batch])  # b aug c h w
        assert all_images.shape[1] == 1
        fg = all_images[:, 0, ...]
        bg = torch.clone(all_images[:, 0, ...])
        bg = bg[torch.randperm(len(bg))]
        bg_clean = torch.clone(bg)
        
        #fg_clone = torch.clone(fg[:, :3, ...])
        t0 = time.time()
        alt_fg, blend_scale = self.blend_batch(fg, bg)
        t1 = time.time()
        print(t1-t0)
        fg = torch.stack([self.transform(i) for i in fg[:, :3, ...]])
        alt_fg = torch.stack([self.transform(i) for i in alt_fg])
        all_im = torch.stack((fg, alt_fg), dim=1)

        #torch.save({"blend_scale": blend_scale, "fg": fg, "bg": bg, "alt_fg": alt_fg, "fg_clean": fg_clone}, "out.pt")
        #exit(1)
        return {
            "image": all_im,
            "label": torch.stack([i["label"] for i in raw_batch]),
            "path": [i["path"] for i in raw_batch],
            "blend_scale": blend_scale,
            "bg":bg_clean,
        }


In [ ]:
scbc = SingleCellBlendedCollator(kernel_size=17, sigma=17, min_val_sigma=.5, strong_transforms=lambda x:x)

In [ ]:
collated_batch = scbc(fg_im)

In [ ]:
for im,bg,bs in zip(collated_batch["image"], collated_batch["bg"], collated_batch["blend_scale"]):
    fig, axs = plt.subplots(1,4)
    axs[0].imshow(einops.rearrange(im[0,:3, ...], "c h w -> h w c").numpy())
    axs[1].imshow(einops.rearrange(im[1,:3, ...], "c h w -> h w c").numpy())
    axs[2].imshow(einops.rearrange(bg[:3, ...], "c h w -> h w c").numpy())
    axs[3].imshow(bs[0,...])
    
    for ax in axs: ax.axis("off")

In [ ]:
for i,(im,bg,bs) in enumerate(zip(collated_batch["image"], collated_batch["bg"], collated_batch["blend_scale"])):
    Image.fromarray(einops.rearrange(im[0,:3, ...], "c h w -> h w c").numpy()).save(f"fg{i}.png")  
    Image.fromarray(einops.rearrange(im[1,:3, ...], "c h w -> h w c").numpy()).save(f"re{i}.png") 
    Image.fromarray(einops.rearrange(bg[:3, ...], "c h w -> h w c").numpy()).save(f"bg{i}.png") 
    Image.fromarray(np.uint8(bs[0,...].numpy()*255), "L").save(f"bs{i}.png") 

In [ ]:
!mkdir bsv2
!mv *.png bsv2

In [ ]:
!tar czvf bsv2.tgz bsv2

In [ ]:
for i, (fg, bg, re, bs) in enumerate(zip(fg_im, bg_im, reconstructed, blend_scale)):
    Image.fromarray(einops.rearrange(re[:3, ...], "c h w -> h w c").numpy()).save(f"re{i}.png")  
    Image.fromarray(einops.rearrange(fg[:3, ...], "c h w -> h w c").numpy()).save(f"fg{i}.png") 
    Image.fromarray(einops.rearrange(bg[:3, ...], "c h w -> h w c").numpy()).save(f"bg{i}.png") 
    Image.fromarray(np.uint8(bs.numpy()*255), "L").save(f"bs{i}.png") 

In [ ]:
plt.imshow(einops.rearrange(reconstructed[5], "c h w -> h w c"))

In [ ]:
bg_nuclei_mask = einops.repeat(bg_im[:,-1,...].to(bool), "b h w -> b c h w", c=3).to(torch.uint8)
bg_nu_fillval = ((bg_im_cutout * (1 - bg_nuclei_mask)).sum(dim=(2,3)) / (1 - bg_nuclei_mask).sum(dim=(2,3))[:,[0]])

In [ ]:
bg_nuclei_mask.sum(dim=(2,3))[:,0]

In [ ]:
to_fill_idx = torch.where(blend_scale>min_in.unsqueeze(1).unsqueeze(1))

bg_im_cutout[:, bg_im[0,-1,...].to(bool)] = bg_im_cutout[:, ~bg_im[0,-1,...].to(bool)].to(float).mean(dim=1).to(torch.uint8).unsqueeze(1)
bg_im_cutout = einops.rearrange(bg_im_cutout, "c h w -> h w c").numpy()


In [ ]:
blend_scale[blend_scale>min_in] = min_in
blend_scale = blend_scale - blend_scale.min()
blend_scale = blend_scale / blend_scale.max()

In [ ]:
bg_im_cutout = torch.clone(bg_im[0,:-1,...])
bg_im_cutout[:, bg_im[0,-1, ...].to(bool)] = bg_im_cutout[:, ~bg_im[0,-1,...].to(bool)].to(float).mean(dim=1).to(torch.uint8).unsqueeze(1)
bg_im_cutout = einops.rearrange(bg_im_cutout, "c h w -> h w c").numpy()


In [ ]:
x,y = np.meshgrid(np.arange(64), np.arange(64))

In [ ]:
circle_mask = (x-32) * (x-32) + (y-32) * (y-32) < 20*20

In [ ]:
for idx in rand_idxs[:50]:
    fg_im = dm.train_dataset_.__getitem__(idx[0])["image"]
    bg_im = dm.train_dataset_.__getitem__(idx[1])["image"]

    
    cutout_mask = torchvision.transforms.functional.gaussian_blur(fg_im[0,[-1],...],kernel_size=21,sigma=5)
    min_in = cutout_mask[fg_im[0,-1,...].astype(bool)].min()
    cutout_mask[cutout_mask>min_in] = min_in
    
    #cutout_mask = cv2.GaussianBlur(circle_mask.astype(float),(21,21),5)
    
    cutout_mask = cutout_mask - cutout_mask.min()
    cutout_mask = cutout_mask / cutout_mask.max()

    bg_im_cutout = torch.clone(bg_im[0,:3,...])
    bg_im_cutout[:, bg_im[0,-1,...].to(bool)] = bg_im_cutout[:, ~bg_im[0,-1,...].to(bool)].to(float).mean(dim=1).to(torch.uint8).unsqueeze(1)
    bg_im_cutout = einops.rearrange(bg_im_cutout, "c h w -> h w c").numpy()

    reconstructed = einops.rearrange(fg_im[0,:3,...], "c h w -> h w c").numpy() * np.expand_dims(cutout_mask, 2) + bg_im_cutout * np.expand_dims(1-cutout_mask, 2)

    fig, axs = plt.subplots(1,4)
    axs[0].imshow(reconstructed.astype(np.uint8))
    axs[1].imshow(einops.rearrange(fg_im[0,:3,...], "c h w -> h w c").numpy())
    axs[2].imshow(einops.rearrange(bg_im[0,:3,...], "c h w -> h w c").numpy())
    axs[3].imshow(cutout_mask)
    
    for ax in axs: ax.axis("off")

In [ ]:
len(dm.train_dataset_)

In [ ]:
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
def to_uint8_func(x):                                               
                 im = (adjust_brightness(adjust_contrast(x[:3,...], 2), 3) * 255).to(
                     torch.uint8)                                                
                 #mask = (x[[3], ...] * 255).to(torch.uint8)                      
                 #return torch.cat((im, mask))   
                 return im

In [ ]:
data2 = torch.load("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/train/out.pt")

In [ ]:
data2.keys()

In [ ]:
for fg, bg, re, fg_clean, bs in zip(data2["fg"], data2["bg_clean"], data2["alt_fg"], data2["fg_clean"], data2["blend_scale"]):   
    fig, axs = plt.subplots(1,5)
    axs[0].imshow(einops.rearrange(to_uint8_func(fg[:3, ...]), "c h w -> h w c").numpy())
    axs[1].imshow(einops.rearrange(to_uint8_func(re[:3, ...]), "c h w -> h w c").numpy())
    axs[2].imshow(einops.rearrange(to_uint8_func(fg_clean[:3, ...]), "c h w -> h w c").numpy())
    axs[3].imshow(einops.rearrange(to_uint8_func(bg[:3, ...]), "c h w -> h w c").numpy())
    axs[4].imshow(bs[0,...])
    
    for ax in axs: ax.axis("off")

In [ ]:
data2["fg"].shape

In [ ]:
data2["fg"].shape

In [ ]:
blend_scale[7]

In [ ]:
test = np.zeros((300,300,3))

In [ ]:
test.shape

In [ ]:
einops.rearrange(test, "h w c -> c h w").shape